In [0]:
from pyspark.sql import functions as F

df_silver_taxi = spark.table("databrickstask.silver.taxi")
df_silver_drivers = spark.table("databrickstask.silver.drivers")
df_silver_users = spark.table("databrickstask.silver.users")

df_gold_taxi = (
    df_silver_taxi
    .groupBy("PULocationID", "description").agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
        )
)
df_gold_drivers = (
    df_silver_drivers
    .groupBy("experience").agg(
        F.countDistinct("name", "car_number").alias("drivers_count"),
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        )

)
df_gold_users = (
    df_silver_users
    .groupBy("registration_date").agg(
        F.count("*").alias("users_count")
        )
    .orderBy(F.desc("users_count"))
    )


(
    df_gold_taxi.write
    .format("delta")
    .mode("overwrite")
    .clusterBy("PULocationID", "description")
    .saveAsTable("databrickstask.gold.taxi_metrics")
)
(
    df_gold_drivers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("databrickstask.gold.driver_metrics")
)

(
    df_gold_users.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("databrickstask.gold.user_metrics")
)

